# 01. Podstawy GoogleTest

## Cele

Po tym notebooku potrafisz:

* napisać test w GoogleTest i uruchomić go przez CMake/CTest,
* dobrać asercję i świadomie wybrać między `EXPECT_*` a `ASSERT_*`,
* poprawnie porównywać liczby zmiennoprzecinkowe,
* sprawdzić, że kod rzuca (albo nie rzuca) wyjątek,
* wybrać przypadki testowe metodą klas równoważności i wartości brzegowych.

Nie omawiamy tu jeszcze testów sparametryzowanych (notebook 2), fixture'ów (notebook 3) ani mocków (notebook 4).

## Pierwszy test

Test w GoogleTest to funkcja zdefiniowana makrem `TEST(NazwaZestawu, NazwaTestu)`
([Primer — Simple Tests](https://google.github.io/googletest/primer.html#simple-tests)).
Pierwszy argument grupuje testy w **zestaw** (*test suite*), drugi nazywa konkretny test.
Obie nazwy muszą być poprawnymi identyfikatorami C++ i nie powinny zawierać podkreślnika `_`.

```cpp
#include <algorithm>
#include <string>

#include <gtest/gtest.h>

std::string reversed(std::string text) {
    std::reverse(text.begin(), text.end());
    return text;
}

TEST(ReversedTest, ReversesWord) {
    EXPECT_EQ(reversed("kajak"), "kajak");
    EXPECT_EQ(reversed("abc"), "cba");
}

TEST(ReversedTest, EmptyStringStaysEmpty) {
    EXPECT_EQ(reversed(""), "");
}
```

Nie piszemy funkcji `main` — dostarcza ją biblioteka `GTest::gtest_main`
([Primer — Writing the main() Function](https://google.github.io/googletest/primer.html#writing-the-main-function)).
W CMake wystarczy zbudować program testowy, dołączyć tę bibliotekę i zarejestrować testy w CTest:

```cmake
add_executable(reversed_tests reversed_test.cpp)
target_link_libraries(reversed_tests PRIVATE GTest::gtest_main)
gtest_discover_tests(reversed_tests)   # each TEST becomes a separate CTest test
```

`gtest_discover_tests` pochodzi z modułu [GoogleTest w CMake](https://cmake.org/cmake/help/latest/module/GoogleTest.html).
Program testowy można też uruchomić bezpośrednio — wtedy wypisze raport GoogleTest dla wszystkich testów.

## Asercje

Asercja sprawdza jeden warunek. Jeśli warunek nie jest spełniony, GoogleTest wypisuje plik, linię
i obie porównywane wartości, a test zostaje oznaczony jako nieudany
([Assertions Reference](https://google.github.io/googletest/reference/assertions.html)).

| Asercja | Sprawdza |
|---|---|
| `EXPECT_TRUE(c)` / `EXPECT_FALSE(c)` | warunek jest prawdziwy / fałszywy |
| `EXPECT_EQ(a, b)` / `EXPECT_NE(a, b)` | `a == b` / `a != b` |
| `EXPECT_LT`, `EXPECT_LE`, `EXPECT_GT`, `EXPECT_GE` | `<`, `<=`, `>`, `>=` |
| `EXPECT_STREQ(a, b)` | zawartość dwóch napisów C (`const char*`); dla `std::string` wystarczy `EXPECT_EQ` |
| `EXPECT_DOUBLE_EQ`, `EXPECT_NEAR` | liczby zmiennoprzecinkowe — patrz niżej |
| `EXPECT_THROW`, `EXPECT_NO_THROW` | wyjątki — patrz niżej |

Do każdej asercji można dopisać własny komunikat operatorem `<<`, np.
`EXPECT_TRUE(isValid(input)) << "input = " << input;`.
Wygodniejsze niż `EXPECT_TRUE(a == b)` jest `EXPECT_EQ(a, b)` — przy błędzie zobaczysz obie wartości,
a nie tylko „false”.

## `EXPECT_*` czy `ASSERT_*`?

Każda asercja ma dwie wersje
([Primer — Assertions](https://google.github.io/googletest/primer.html#assertions)):

* **`EXPECT_*`** zgłasza *niekrytyczne* niepowodzenie (*nonfatal failure*): błąd jest zapisywany,
  a funkcja **wykonuje się dalej**. Jeden przebieg testu pokaże więc wszystkie niespełnione oczekiwania.
* **`ASSERT_*`** zgłasza *krytyczne* niepowodzenie (*fatal failure*) i **natychmiast kończy bieżącą
  funkcję** (wykonuje `return`). Kod po nieudanej asercji `ASSERT_*` w tej funkcji już się nie wykona.

W obu przypadkach test zostaje oznaczony jako nieudany, a każde niepowodzenie jest raportowane osobno.
Domyślnie używaj `EXPECT_*`. `ASSERT_*` stosuj wtedy, gdy dalsze sprawdzanie nie ma sensu albo byłoby
niebezpieczne — np. gdy kolejne linie zakładają, że wektor ma odpowiedni rozmiar:

```cpp
#include <vector>

#include <gtest/gtest.h>

std::vector<int> firstSquares(int n) {
    std::vector<int> result;
    for (int i = 1; i <= n; ++i) {
        result.push_back(i * i);
    }
    return result;
}

TEST(FirstSquaresTest, ReturnsSquaresInOrder) {
    const auto squares = firstSquares(3);

    ASSERT_EQ(squares.size(), 3u);  // without it, squares[2] could read out of bounds
    EXPECT_EQ(squares[0], 1);
    EXPECT_EQ(squares[1], 4);
    EXPECT_EQ(squares[2], 9);
}
```

Uwaga: `ASSERT_*` kończy **tylko bieżącą funkcję**, a nie cały test. Jeśli wywołasz ją w funkcji
pomocniczej, przerwana zostanie funkcja pomocnicza, a test będzie kontynuowany od miejsca jej wywołania
([Advanced — Propagating Fatal Failures](https://google.github.io/googletest/advanced.html#propagating-fatal-failures)).
Z tego samego powodu (`ASSERT_*` to w środku `return;`) asercji krytycznych można używać tylko
w funkcjach zwracających `void`
([Advanced — Assertion Placement](https://google.github.io/googletest/advanced.html#assertion-placement)).

```cpp
#include <iostream>

#include <gtest/gtest.h>

void checkPositive(int value) {
    ASSERT_GT(value, 0);
    std::cout << "helper: " << value << " is positive\n";  // skipped when the assertion fails
}

TEST(HelperDemo, FatalFailureEndsOnlyTheHelper) {
    checkPositive(-1);
    std::cout << "test body continues\n";  // still printed
}
```

## Liczby zmiennoprzecinkowe

Typy `float` i `double` przechowują liczby w systemie dwójkowym (standard IEEE 754). Większości ułamków
dziesiętnych — 0,1, 0,2 czy 0,3 — nie da się tak zapisać dokładnie, tak jak 1/3 nie da się zapisać
dokładnie w systemie dziesiętnym. Komputer przechowuje najbliższą możliwą wartość, a błędy zaokrągleń
mogą się sumować ([The Floating-Point Guide](https://floating-point-gui.de/basic/),
[0.30000000000000004.com](https://0.30000000000000004.com/)).

Te błędy są **deterministyczne**: to samo działanie na tych samych danych daje za każdym razem
dokładnie ten sam wynik. Problem polega na tym, że ten wynik nie jest tym, którego oczekujemy „na papierze”.

`EXPECT_EQ` porównuje wartości **dokładnie** (operatorem `==`), więc do liczb zmiennoprzecinkowych
służą osobne asercje
([Assertions Reference — Floating-Point Comparison](https://google.github.io/googletest/reference/assertions.html#floating-point)):

| Asercja | Sprawdza |
|---|---|
| `EXPECT_DOUBLE_EQ(a, b)` (`EXPECT_FLOAT_EQ` dla `float`) | `a` i `b` różnią się o co najwyżej 4 ULP — czyli są „prawie równe” w skali tych liczb |
| `EXPECT_NEAR(a, b, abs_error)` | `abs(a - b) <= abs_error` — tolerancję bezwzględną podajesz sam |

ULP (*unit in the last place*) to odległość między dwiema sąsiednimi liczbami, które da się zapisać
w danym typie. `EXPECT_DOUBLE_EQ` dopasowuje więc tolerancję do wielkości porównywanych liczb.
`EXPECT_NEAR` przydaje się, gdy wiesz, jaka dokładność jest wymagana (np. „do 0,01 mm”).

Uruchom demonstrację — plik [`demos/float_demo.cpp`](demos/float_demo.cpp) zawiera trzy testy,
z których jeden celowo nie przechodzi. Przeczytaj, jakie wartości wypisał GoogleTest.

In [ ]:
# Build and run the floating-point demo (one test fails on purpose)
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug --output-on-failure

Dla dociekliwych: [B. Dawson — *Comparing Floating Point Numbers*](https://randomascii.wordpress.com/2012/02/25/comparing-floating-point-numbers-2012-edition/)
(dlaczego stały „epsilon” nie wystarcza) oraz klasyczny artykuł
[D. Goldberg — *What Every Computer Scientist Should Know About Floating-Point Arithmetic*](https://docs.oracle.com/cd/E19957-01/806-3568/ncg_goldberg.html).
Po polsku: [IEEE 754 w Wikipedii](https://pl.wikipedia.org/wiki/IEEE_754) i rozdział
[„Arytmetyka liczb zmiennoprzecinkowych: problemy i ograniczenia”](https://docs.python.org/pl/3/tutorial/floatingpoint.html)
z oficjalnego tutoriala Pythona (problem jest ten sam w każdym języku).

## Wyjątki

Jeśli zachowaniem funkcji jest rzucenie wyjątku, to też trzeba to przetestować
([Assertions Reference — Exception Assertions](https://google.github.io/googletest/reference/assertions.html#exceptions)):

| Asercja | Sprawdza |
|---|---|
| `EXPECT_THROW(instrukcja, typ)` | instrukcja rzuca wyjątek podanego typu (lub typu pochodnego) |
| `EXPECT_ANY_THROW(instrukcja)` | instrukcja rzuca dowolny wyjątek |
| `EXPECT_NO_THROW(instrukcja)` | instrukcja nie rzuca żadnego wyjątku |

```cpp
#include <stdexcept>
#include <string>

#include <gtest/gtest.h>

TEST(StoiTest, ThrowsInvalidArgumentForNonNumber) {
    EXPECT_THROW(std::stoi("abc"), std::invalid_argument);
    EXPECT_THROW(std::stoi("99999999999999999999"), std::out_of_range);
    EXPECT_NO_THROW(std::stoi("42"));
}
```

Warto testować możliwie konkretny typ wyjątku: `EXPECT_ANY_THROW` przejdzie także wtedy, gdy kod rzuci
zupełnie inny wyjątek niż zamierzony. Treść komunikatu wyjątku można sprawdzić matcherem
[`ThrowsMessage`](https://google.github.io/googletest/reference/matchers.html#exception-matchers).

## Jakie przypadki testować? Klasy równoważności i wartości brzegowe

Nie da się przetestować wszystkich możliwych wejść. Dwie podstawowe techniki pomagają wybrać
niewiele przypadków, które razem dużo sprawdzają
([sylabus ISTQB CTFL 4.0, §4.2.1–4.2.2](https://sjsi.org/download/13814/)):

* **Podział na klasy równoważności** — dzielimy dane wejściowe na klasy, w których program powinien
  zachowywać się tak samo. Wystarczy jeden test na klasę. Klasy obejmują zarówno dane poprawne,
  jak i **niepoprawne**.
* **Analiza wartości brzegowych** — błędy najczęściej pojawiają się na granicach klas (`<` zamiast `<=`,
  przesunięcie o jeden). Testujemy więc wartości tuż przy granicach: ostatnią wartość jednej klasy
  i pierwszą następnej.

Przykład: funkcja zamienia liczbę punktów (0–100) na ocenę.

| Klasa | Punkty | Ocena | Wartości brzegowe |
|---|---|---|---|
| niepoprawna | < 0 | wyjątek | −1 |
| poprawna | 0–49 | 2 | 0, 49 |
| poprawna | 50–69 | 3 | 50, 69 |
| poprawna | 70–89 | 4 | 70, 89 |
| poprawna | 90–100 | 5 | 90, 100 |
| niepoprawna | > 100 | wyjątek | 101 |

```cpp
#include <stdexcept>

#include <gtest/gtest.h>

int grade(int points) {
    if (points < 0 || points > 100) {
        throw std::out_of_range("points must be in [0, 100]");
    }
    if (points < 50) return 2;
    if (points < 70) return 3;
    if (points < 90) return 4;
    return 5;
}

TEST(GradeTest, BoundaryBetweenFailAndPass) {
    EXPECT_EQ(grade(49), 2);
    EXPECT_EQ(grade(50), 3);
}

TEST(GradeTest, RejectsPointsOutsideRange) {
    EXPECT_THROW(grade(-1), std::out_of_range);
    EXPECT_THROW(grade(101), std::out_of_range);
}
```

Pełna tabela to dziesięć wartości brzegowych. Pisanie dla każdej osobnej asercji szybko staje się
żmudne — w notebooku 2 zobaczysz, jak zamienić taką tabelę bezpośrednio w testy.

Historia o wartościach brzegowych: przez ok. 20 lat niemal każda implementacja wyszukiwania binarnego
zawierała błąd w linii `int mid = (low + high) / 2;` — przepełnienie pojawiało się dopiero dla
ogromnych tablic ([J. Bloch, Google Research](https://research.google/blog/extra-extra-read-all-about-it-nearly-all-binary-searches-and-mergesorts-are-broken/)).

## Ćwiczenie

*Treść ćwiczenia — w przygotowaniu.*

## Linki i materiały

* [GoogleTest Primer](https://google.github.io/googletest/primer.html) — podstawy: testy, asercje, uruchamianie
* [Assertions Reference](https://google.github.io/googletest/reference/assertions.html) — pełna lista asercji
* [Advanced — Propagating Fatal Failures](https://google.github.io/googletest/advanced.html#propagating-fatal-failures) — `ASSERT_*` w funkcjach pomocniczych
* [The Floating-Point Guide](https://floating-point-gui.de/) — przystępnie o liczbach zmiennoprzecinkowych
* [Sylabus ISTQB CTFL 4.0 (PL, PDF)](https://sjsi.org/download/13814/) — rozdz. 4.2: klasy równoważności, wartości brzegowe

---

[Strona główna](../_start.ipynb) | [02. Testy sparametryzowane →](../02_parametrized/02_parametrized.ipynb)